# 1 RNN

# 1 RNN(LSTM)을 사용한 텍스트 분류

## 1) 공통

In [1]:
import os
import random
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torch.optim as optim
from torch.optim.lr_scheduler import StepLR

from tqdm.auto import tqdm

c:\Users\kevin\anaconda3\envs\campus\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import matplotlib.pyplot as plt

def plot_graphs(history, metric):
    plt.plot(history[metric])
    plt.xlabel("Epochs")
    plt.ylabel(metric)
    plt.show()

## 2)  IMDB 데이터셋 (긍정/부정) 다운로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
imdb_dir = '/content/drive/MyDrive/딥러닝NLP/data/IMDB Dataset.csv'

In [3]:
df = pd.read_csv('IMDB Dataset.csv')
df.head(5)

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df.drop_duplicates(inplace=True)

In [5]:
df

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive
...,...,...
49995,I thought this movie did a down right good job...,positive
49996,"Bad plot, bad dialogue, bad acting, idiotic di...",negative
49997,I am a Catholic taught in parochial elementary...,negative
49998,I'm going to have to disagree with the previou...,negative


## 3) 학습데이터 준비

다음은 문장을 전처리하여 숫자화하고 padding을 추가하여 문자열을 encoding하고,
 이를 dataset, dataloader로 만들어 학습을 위한 `batch`로 만들어준다.

In [6]:
tokenized=[]

stopwords = [l.rstrip() for l in open('./stopwords.txt', 'r').readlines()]
import re
for sentence in df['review']:
    sentence = re.sub('<br />', ' ', sentence)
    sentence = re.sub('[^\w\s\'\"\&]+', '', sentence)
    temp = sentence.lower().split(' ') # 토큰화
    temp = [word for word in temp if not word in stopwords] # 불용어 제거
    tokenized.append(list(filter(None, temp)))
print(tokenized[:10])

<>:7: SyntaxWarning: invalid escape sequence '\w'
<>:7: SyntaxWarning: invalid escape sequence '\w'
C:\Users\kevin\AppData\Local\Temp\ipykernel_33776\3905645626.py:7: SyntaxWarning: invalid escape sequence '\w'
  sentence = re.sub('[^\w\s\'\"\&]+', '', sentence)


[['reviewers', 'mentioned', 'watching', '1', 'oz', 'episode', 'hooked', 'happened', 'struck', 'oz', 'brutality', 'unflinching', 'scenes', 'violence', 'set', 'word', 'trust', 'faint', 'hearted', 'timid', 'pulls', 'punches', 'drugs', 'sex', 'violence', 'hardcore', 'classic', 'word', 'called', 'oz', 'nickname', 'oswald', 'maximum', 'security', 'penitentary', 'focuses', 'emerald', 'city', 'experimental', 'prison', 'cells', 'glass', 'fronts', 'inwards', 'privacy', 'agenda', 'em', 'city', 'manyaryans', 'muslims', 'gangstas', 'latinos', 'christians', 'italians', 'irish', 'moreso', 'scuffles', 'death', 'stares', 'dodgy', 'dealings', 'shady', 'agreements', 'main', 'appeal', 'forget', 'pretty', 'pictures', 'painted', 'mainstream', 'audiences', 'forget', 'charm', 'forget', 'romanceoz', 'mess', 'episode', 'struck', 'nasty', 'surreal', 'ready', 'watched', 'developed', 'taste', 'oz', 'accustomed', 'levels', 'graphic', 'violence', 'violence', 'injustice', 'crooked', 'guards', 'sold', 'nickel', 'inmat

In [ ]:
from nltk import FreqDist
vocab = FreqDist(np.hstack(tokenized))

In [ ]:
len(vocab)

In [ ]:
vocab_size = 8190 # +2개는 PAD, UNK
vocab = vocab.most_common(vocab_size)

word_to_index = {word[0] : index + 2 for index, word in enumerate(vocab)}
word_to_index['<PAD>'] = 1
word_to_index['<UNK>'] = 0

In [ ]:
encoded = []
for line in tokenized: #입력 데이터에서 1줄씩 문장을 읽음
    temp = []
    for w in line: #각 줄에서 1개씩 글자를 읽음
      try:
        temp.append(word_to_index[w]) # 글자를 해당되는 정수로 변환
      except KeyError: # 단어 집합에 없는 단어일 경우 UNK로 대체된다.
        temp.append(word_to_index['<UNK>']) # UNK의 인덱스로 변환

    encoded.append(temp)

print(encoded[:10])

In [ ]:
max_len = max(len(l) for l in encoded)
print('리뷰의 최대 길이 : %d' % max_len)
print('리뷰의 최소 길이 : %d' % min(len(l) for l in encoded))
print('리뷰의 평균 길이 : %f' % (sum(map(len, encoded))/len(encoded)))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.displot([len(l) for l in encoded], kde=True)
# 텍스트 데이터를 처리할 경우 문장단위로 분리하는게 좋음
plt.axvline(x=256, color="red")

In [ ]:
max_len = 256
for line in encoded:
    if len(line) < max_len: # 현재 샘플이 정해준 길이보다 짧으면
        line += [word_to_index['<PAD>']] * (max_len - len(line)) # 나머지는 전부 'pad' 토큰으로 채운다.
print('리뷰의 최대 길이 : %d' % max(len(l) for l in encoded))
print('리뷰의 최소 길이 : %d' % min(len(l) for l in encoded))
print('리뷰의 평균 길이 : %f' % (sum(map(len, encoded))/len(encoded)))

##########################################################################################
# 전처리 함수 만들기

In [ ]:
# 길이 제한, 위의 데이터 자르기
encoded = [line[:max_len] for line in encoded]
sum(map(len, encoded))/len(encoded)

In [ ]:
y = df["sentiment"].apply(lambda x: 1 if x == 'positive' else 0).values

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(encoded, y,test_size=0.2,stratify=y, shuffle=True)

In [ ]:
# 데이터셋 만들기
from torch.utils.data import Dataset
class TextDataset(Dataset):
    def __init__(self, x, y):
        self.x = torch.tensor(x, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float)

    def __getitem__(self, index):
        # x = torch.from_numpy(self.x[index]).float()
        return self.x[index], self.y[index]

    def __len__(self):
        return len(self.x)

train_dataset = TextDataset(X_train, y_train)
test_dataset = TextDataset(X_test, y_test)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=64)

### AutoTokenizer

※ 참고: 나중에 pretrained model (bert, electra, gpt 등) 사용시에 적용

In [ ]:
%pip install transformers

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-cased")

In [ ]:
tokenizer("Using a Transformer network is simple")

In [ ]:
tokens = tokenizer.tokenize("Using a Transformer network is simple")
input_ids = tokenizer.convert_tokens_to_ids(tokens)

In [ ]:
tokenizer.decode(input_ids)

## 4) 모델 정의

In [ ]:
class LSTMClassification(nn.Module):
    def __init__(self,
                 num_classes, # 클래스 개수 (긍정 1, 부정 0)
                 embed_size=128, # 단어 임베딩 사이즈
                 vocab_size=8192, # 단어 vocab size
                 sequence_length=256 # 문장 길이 (128)
                 ):
        super(LSTMClassification, self).__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size,
                               embedding_dim=embed_size,
                               padding_idx=1)

        self.lstm = nn.LSTM(input_size=embed_size, hidden_size=128,
                           num_layers=1, batch_first=True)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.embedding(x)
        output, (hidden, cell) = self.lstm(x)
        output = self.classifier(hidden[-1])
        return output

model = LSTMClassification(1, )

### Quiz.

만약, 긍정, 부정 2진분류 문제가 아닌 긍정, 부정, 중립 3진분류 문제일 경우
1. 모델의 layer를 어떻게 수정해야 하는지 (Linear Layer)
2. loss 함수는 어떤 것을 사용해야 하는지 (multi class loss)
3. accuracy 계산은 어떻게 해야하는지 (hint: argmax)


In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

criterion = nn.BCELoss().cuda()
optimizer = optim.Adam(model.parameters(), lr=0.01)
scheduler = StepLR(optimizer, step_size=7, gamma=0.7)

## 5) 모델 학습

In [ ]:
# 모델 학습시키기
epochs = 10

history = {
    'loss': [],
    'val_loss': [],
    'val_acc': []
}

for epoch in range(1, epochs+1):
    # 학습
    model.train()
    train_loss = []
    for batch_idx, (data, label) in enumerate(tqdm(iter(train_loader))):
        data, label = data.to(device), label.to(device)
        optimizer.zero_grad()
        target = model(data).reshape(-1)
        loss = criterion(target, label)

        loss.backward()
        optimizer.step()
        train_loss.append(loss.item())

    _train_loss = np.mean(train_loss)
    history['loss'].append(_train_loss)

    # 테스트
    model.eval()
    val_loss = []
    correct = 0
    for data, label in tqdm(iter(test_loader)):
        data, label = data.to(device), label.to(device)
        with torch.no_grad():
            target = model(data).reshape(-1).detach()
        loss = criterion(target, label)
        val_loss.append(loss.item())
        correct += (target>0.5).type(torch.int).eq(label.view_as(target)).sum().item()

    _val_loss = np.mean(val_loss)
    history['val_loss'].append(_val_loss)
    history['val_acc'].append(correct/len(test_loader.dataset))

    if epoch % 1 == 0:
        print(f'Epoch [{epoch}], Train Loss : [{_train_loss:.5f}] Val Loss : [{_val_loss:.5f}] Val Accuracy : [{correct/len(test_loader.dataset):.4f}]')

    scheduler.step()

## 6) 예측 및 평가

In [ ]:
# predict on a sample text with positive, negative words

sample_pred_text = ('The movie was cool. The animation and the graphics '
                    'were out of this world. I would recommend this movie.')


sample_pred_text = ('The movie was not good. The animation and the graphics '
                    'were terrible. I would not recommend this movie.')

tokenized=[]

# for sentence in df['review']:
sample_pred_text = re.sub('<br />', ' ', sample_pred_text)
sample_pred_text = re.sub('[^\w\s\'\"\&]+', '', sample_pred_text)
temp = sample_pred_text.lower().split(' ') # 토큰화
temp = [word for word in temp if not word in stopwords] # 불용어 제거
tokenized.append(list(filter(None, temp)))

encoded = []
for line in tokenized: #입력 데이터에서 1줄씩 문장을 읽음
    temp = []
    for w in line: #각 줄에서 1개씩 글자를 읽음
        try:
            temp.append(word_to_index[w]) # 글자를 해당되는 정수로 변환
        except KeyError: # 단어 집합에 없는 단어일 경우 UNK로 대체된다.
            temp.append(word_to_index['<UNK>']) # UNK의 인덱스로 변환
    encoded.append(temp)

max_len = 256
for line in encoded:
    if len(line) < max_len: # 현재 샘플이 정해준 길이보다 짧으면
        line += [word_to_index['<PAD>']] * (max_len - len(line)) # 나머지는 전부 'pad' 토큰으로 채운다.


In [ ]:
tokenized

In [ ]:
encoded

In [ ]:
pred_dataset = TextDataset(encoded, [0])
pred_loader = torch.utils.data.DataLoader(pred_dataset, batch_size=1)

In [ ]:
# 테스트
model.eval()
val_loss = []
correct = 0
for data, _ in tqdm(iter(pred_loader)):
    data = data.to(device)
    with torch.no_grad():
        target = model(data).reshape(-1).detach()
    prediction = ["positive" if (t>0.5).type(torch.int) else "negative" for t in target]

print(prediction)

In [ ]:
# 위의 내용을 함수화
def make_dataloader(sample_text):
    tokenized=[]

    sample_text = re.sub('<br />', ' ', sample_text)
    sample_text = re.sub('[^\w\s\'\"\&]+', '', sample_text)
    temp = sample_text.lower().split(' ') # 토큰화
    temp = [word for word in temp if not word in stopwords] # 불용어 제거
    tokenized.append(list(filter(None, temp)))

    encoded = []
    for line in tokenized: #입력 데이터에서 1줄씩 문장을 읽음
        temp = []
        for w in line: #각 줄에서 1개씩 글자를 읽음
            try:
                temp.append(word_to_index[w]) # 글자를 해당되는 정수로 변환
            except KeyError: # 단어 집합에 없는 단어일 경우 UNK로 대체된다.
                temp.append(word_to_index['<UNK>']) # UNK의 인덱스로 변환
        encoded.append(temp)

    max_len = 256
    for line in encoded:
        if len(line) < max_len: # 현재 샘플이 정해준 길이보다 짧으면
            line += [word_to_index['<PAD>']] * (max_len - len(line)) # 나머지는 전부 'pad' 토큰으로 채운다.

    pred_dataset = TextDataset(encoded, [0])
    pred_loader = torch.utils.data.DataLoader(pred_dataset, batch_size=1)

    return pred_loader


In [ ]:
def sample_prediction(sample_text):
    pred_loader = make_dataloader(sample_text)
    # 테스트
    model.eval()
    for data, _ in tqdm(iter(pred_loader)):
        data = data.to(device)
        with torch.no_grad():
            target = model(data).reshape(-1).detach()
        prediction = ["positive" if (t>0.5).type(torch.int) else "negative" for t in target]

    return prediction

In [ ]:
plot_graphs(history, 'loss')

In [ ]:
plot_graphs(history, 'val_acc')

---

# 2 GRU

## 1) 모델 정의

In [ ]:
class GRUClassification(nn.Module):
    def __init__(self,
                 num_classes, # 클래스 개수 (긍정 1, 부정 0)
                 embed_size=128, # 단어 임베딩 사이즈
                 vocab_size=8192, # 단어 vocab size
                 sequence_length=256 # 문장 길이 (128)
                 ):
        super(GRUClassification, self).__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size,
                               embedding_dim=embed_size,
                               padding_idx=1)

        self.gru = nn.GRU(input_size=embed_size, hidden_size=128,
                           num_layers=1, batch_first=True)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.embedding(x)
        output, hidden = self.gru(x)
        output = self.classifier(hidden[-1])
        return output

model = GRUClassification(1, )

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

criterion = nn.BCELoss().cuda()
optimizer = optim.Adam(model.parameters(), lr=0.01)
scheduler = StepLR(optimizer, step_size=7, gamma=0.7)

## 2) 모델 학습

In [ ]:
# 모델 학습시키기
epochs = 10

history = {
    'loss': [],
    'val_loss': [],
    'val_acc': []
}

for epoch in range(1, epochs+1):
    # 학습
    model.train()
    train_loss = []
    for batch_idx, (data, label) in enumerate(tqdm(iter(train_loader))):
        data, label = data.to(device), label.to(device)
        optimizer.zero_grad()
        target = model(data).reshape(-1)
        loss = criterion(target, label)

        loss.backward()
        optimizer.step()
        train_loss.append(loss.item())

    _train_loss = np.mean(train_loss)
    history['loss'].append(_train_loss)

    # 테스트
    model.eval()
    val_loss = []
    correct = 0
    for data, label in tqdm(iter(test_loader)):
        data, label = data.to(device), label.to(device)
        with torch.no_grad():
            target = model(data).reshape(-1).detach()
        loss = criterion(target, label)
        val_loss.append(loss.item())
        correct += (target>0.5).type(torch.int).eq(label.view_as(target)).sum().item()

    _val_loss = np.mean(val_loss)
    history['val_loss'].append(_val_loss)
    history['val_acc'].append(correct/len(test_loader.dataset))

    if epoch % 1 == 0:
        print(f'Epoch [{epoch}], Train Loss : [{_train_loss:.5f}] Val Loss : [{_val_loss:.5f}] Val Accuracy : [{correct/len(test_loader.dataset):.4f}]')

    scheduler.step()

## 3) 예측 및 평가

In [ ]:
# predict on a sample text with positive words.

sample_pred_text = ('The movie was cool. The animation and the graphics '
                    'were out of this world. I would recommend this movie.')

predictions = sample_prediction(sample_pred_text)
print(predictions)

In [ ]:
# predict on a sample text with negative words

sample_pred_text = ('The movie was not good. The animation and the graphics '
                    'were terrible. I would not recommend this movie.')
predictions = sample_prediction(sample_pred_text)
print(predictions)

In [ ]:
plot_graphs(history, 'loss')

In [ ]:
plot_graphs(history, 'val_acc')

---

# 3 LSTM Bidirectional

## 1) 모델 정의

In [ ]:
class BLSTMClassification(nn.Module):
    def __init__(self,
                 num_classes, # 클래스 개수 (긍정 1, 부정 0)
                 embed_size=128, # 단어 임베딩 사이즈
                 vocab_size=8192, # 단어 vocab size
                 sequence_length=256 # 문장 길이 (128)
                 ):
        super(BLSTMClassification, self).__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size,
                               embedding_dim=embed_size,
                               padding_idx=1)

        self.lstm = nn.LSTM(input_size=embed_size, hidden_size=128,
                           num_layers=1, bidirectional=True, batch_first=True)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128*2, 1), # bidirectional  *2
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.embedding(x)
        output, (hidden, _) = self.lstm(x)
        output = self.classifier(hidden.reshape(x.shape[0], -1))
        return output

model = BLSTMClassification(1, )

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

criterion = nn.BCELoss().cuda()
optimizer = optim.Adam(model.parameters(), lr=0.01)
scheduler = StepLR(optimizer, step_size=7, gamma=0.7)

## 2) 모델 학습

In [ ]:
# 모델 학습시키기
epochs = 10

history = {
    'loss': [],
    'val_loss': [],
    'val_acc': []
}

for epoch in range(1, epochs+1):
    # 학습
    model.train()
    train_loss = []
    for batch_idx, (data, label) in enumerate(tqdm(iter(train_loader))):
        data, label = data.to(device), label.to(device)
        optimizer.zero_grad()
        target = model(data).reshape(-1)
        loss = criterion(target, label)

        loss.backward()
        optimizer.step()
        train_loss.append(loss.item())

    _train_loss = np.mean(train_loss)
    history['loss'].append(_train_loss)

    # 테스트
    model.eval()
    val_loss = []
    correct = 0
    for data, label in tqdm(iter(test_loader)):
        data, label = data.to(device), label.to(device)
        with torch.no_grad():
            target = model(data).reshape(-1).detach()
        loss = criterion(target, label)
        val_loss.append(loss.item())
        correct += (target>0.5).type(torch.int).eq(label.view_as(target)).sum().item()

    _val_loss = np.mean(val_loss)
    history['val_loss'].append(_val_loss)
    history['val_acc'].append(correct/len(test_loader.dataset))

    if epoch % 1 == 0:
        print(f'Epoch [{epoch}], Train Loss : [{_train_loss:.5f}] Val Loss : [{_val_loss:.5f}] Val Accuracy : [{correct/len(test_loader.dataset):.4f}]')

    scheduler.step()

## 3) 예측 및 평가

In [ ]:
# predict on a sample text with positive words.

sample_pred_text = ('The movie was cool. The animation and the graphics '
                    'were out of this world. I would recommend this movie.')

predictions = sample_prediction(sample_pred_text)
print(predictions)

In [ ]:
# predict on a sample text with negative words

sample_pred_text = ('The movie was not good. The animation and the graphics '
                    'were terrible. I would not recommend this movie.')
predictions = sample_prediction(sample_pred_text)
print(predictions)

In [ ]:
plot_graphs(history, 'val_acc')

In [ ]:
plot_graphs(history, 'loss')

---

# 4 LSTM Stacking

## 1) 모델 정의

In [ ]:
class StackLSTMClassification(nn.Module):
    def __init__(self,
                 num_classes, # 클래스 개수 (긍정 1, 부정 0)
                 embed_size=128, # 단어 임베딩 사이즈
                 vocab_size=8192, # 단어 vocab size
                 sequence_length=256 # 문장 길이 (128)
                 ):
        super(StackLSTMClassification, self).__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size,
                               embedding_dim=embed_size,
                               padding_idx=1)

        self.lstm1 = nn.LSTM(input_size=embed_size, hidden_size=128,
                           num_layers=1, batch_first=True)
        self.lstm2 = nn.LSTM(input_size=128, hidden_size=64,
                           num_layers=1, batch_first=True)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = self.embedding(x)
        output, (hidden, _) = self.lstm1(x)
        output, (hidden, _) = self.lstm2(output)
        output = self.classifier(hidden[-1])
        return output

model = StackLSTMClassification(1, )

In [ ]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

criterion = nn.BCELoss().cuda()
optimizer = optim.Adam(model.parameters(), lr=0.01)
scheduler = StepLR(optimizer, step_size=7, gamma=0.7)

## 2) 모델 학습

In [ ]:
# 모델 학습시키기
epochs = 10

history = {
    'loss': [],
    'val_loss': [],
    'val_acc': []
}

for epoch in range(1, epochs+1):
    # 학습
    model.train()
    train_loss = []
    for batch_idx, (data, label) in enumerate(tqdm(iter(train_loader))):
        data, label = data.to(device), label.to(device)
        optimizer.zero_grad()
        target = model(data).reshape(-1)
        loss = criterion(target, label)

        loss.backward()
        optimizer.step()
        train_loss.append(loss.item())

    _train_loss = np.mean(train_loss)
    history['loss'].append(_train_loss)

    # 테스트
    model.eval()
    val_loss = []
    correct = 0
    for data, label in tqdm(iter(test_loader)):
        data, label = data.to(device), label.to(device)
        with torch.no_grad():
            target = model(data).reshape(-1).detach()
        loss = criterion(target, label)
        val_loss.append(loss.item())
        correct += (target>0.5).type(torch.int).eq(label.view_as(target)).sum().item()

    _val_loss = np.mean(val_loss)
    history['val_loss'].append(_val_loss)
    history['val_acc'].append(correct/len(test_loader.dataset))

    if epoch % 1 == 0:
        print(f'Epoch [{epoch}], Train Loss : [{_train_loss:.5f}] Val Loss : [{_val_loss:.5f}] Val Accuracy : [{correct/len(test_loader.dataset):.4f}]')

    scheduler.step()

## 3) 예측 및 평가

In [ ]:
# predict on a sample text with positive words.

sample_pred_text = ('The movie was cool. The animation and the graphics '
                    'were out of this world. I would recommend this movie.')
predictions = sample_prediction(sample_pred_text)
print(predictions)

In [ ]:
# predict on a sample text with negative words.

sample_pred_text = ('The movie was not good. The animation and the graphics '
                    'were terrible. I would not recommend this movie.')
predictions = sample_prediction(sample_pred_text)
print(predictions)

In [ ]:
plot_graphs(history, 'val_acc')

In [ ]:
plot_graphs(history, 'loss')